# Part 4: Mitigation

In [1]:
import pandas as pd
import torch
import numpy as np
import os
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

print("\n1. Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted")

model_path = '/content/drive/MyDrive/distilbert_toxicity_model_final'

if os.path.exists(model_path):
    print(f"✓ Model found at: {model_path}")
else:
    print(f"❌ Model not found at: {model_path}")
    print("   Checking alternative locations...")

    # Alternative: check if model is in current directory
    if os.path.exists('./distilbert_toxicity_model_final'):
        model_path = './distilbert_toxicity_model_final'
        print(f"✓ Model found locally at: {model_path}")
    else:
        print("   Please ensure model is saved in Drive")

print("\n2. Loading saved data...")

if os.path.exists('train_subset_cleaned.csv') and os.path.exists('eval_subset_cleaned.csv'):
    train_df = pd.read_csv('train_subset_cleaned.csv')
    eval_df = pd.read_csv('eval_subset_cleaned.csv')
    print("   Loaded from current directory")
else:
    train_df = pd.read_csv('/content/drive/MyDrive/train_subset_cleaned.csv')
    eval_df = pd.read_csv('/content/drive/MyDrive/eval_subset_cleaned.csv')
    print("   Loaded from Google Drive")

print(f"   Training set: {len(train_df):,} rows")
print(f"   Evaluation set: {len(eval_df):,} rows")
print(f"   Training toxic %: {train_df['toxic_binary'].mean():.2%}")
print(f"   Evaluation toxic %: {eval_df['toxic_binary'].mean():.2%}")

print("\n3. Loading saved model from Drive...")
model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
print(f"   Model loaded on {device}")
print(f"   Model parameters: {model.num_parameters():,}")

print("\n4. Verification:")
print(f"   Eval toxic %: {eval_df['toxic_binary'].mean():.2%}")
print(f"   Model device: {next(model.parameters()).device}")

print("\nSetup complete! Ready for analysis.")


1. Mounting Google Drive...
Mounted at /content/drive
Drive mounted
✓ Model found at: /content/drive/MyDrive/distilbert_toxicity_model_final

2. Loading saved data...
   Loaded from current directory
   Training set: 100,000 rows
   Evaluation set: 20,000 rows
   Training toxic %: 8.00%
   Evaluation toxic %: 8.00%

3. Loading saved model from Drive...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

   Model loaded on cuda
   Model parameters: 66,955,010

4. Verification:
   Eval toxic %: 8.00%
   Model device: cuda:0

Setup complete! Ready for analysis.


In [3]:
# Part 4: Mitigation Techniques
from sklearn.utils import compute_class_weight
from transformers import Trainer
from sklearn.utils import compute_class_weight
from transformers import Trainer
from datasets import Dataset
import torch
import numpy as np
import pandas as pd
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )
train_df['is_black_cohort'] = (train_df['black'] >= 0.5).astype(int)
train_df['is_white_cohort'] = ((train_df['black'] < 0.1) & (train_df['white'] >= 0.5)).astype(int)

train_df['protected'] = -1
train_df.loc[train_df['is_black_cohort'] == 1, 'protected'] = 1  # Unprivileged
train_df.loc[train_df['is_white_cohort'] == 1, 'protected'] = 0  # Privileged

print(f"\n1. Cohort sizes in training set:")
print(f"   Black cohort (unprivileged): {(train_df['protected'] == 1).sum()} rows")
print(f"   White cohort (privileged): {(train_df['protected'] == 0).sum()} rows")
print(f"   Other: {(train_df['protected'] == -1).sum()} rows")

print("\n2. Computing reweighing weights...")

weights = {}
groups = ['black_cohort', 'white_cohort']
labels = [0, 1]

for label in labels:
    for group in groups:
        if group == 'black_cohort':
            mask = (train_df['toxic_binary'] == label) & (train_df['is_black_cohort'] == 1)
            group_name = 'Black'
        else:
            mask = (train_df['toxic_binary'] == label) & (train_df['is_white_cohort'] == 1)
            group_name = 'White'

        count = mask.sum()
        expected_count = (train_df['toxic_binary'] == label).sum() * (train_df['is_black_cohort' if group == 'black_cohort' else 'is_white_cohort'].sum() / len(train_df))

        if count > 0 and expected_count > 0:
            weight = expected_count / count
        else:
            weight = 1.0

        weights[f'{group_name}_{label}'] = weight
        print(f"   {group_name} cohort, label={label}: weight={weight:.3f} (count={count})")

train_df['sample_weight'] = 1.0
for idx in train_df.index:
    if train_df.loc[idx, 'is_black_cohort'] == 1:
        train_df.loc[idx, 'sample_weight'] = weights[f'Black_{train_df.loc[idx, "toxic_binary"]}']
    elif train_df.loc[idx, 'is_white_cohort'] == 1:
        train_df.loc[idx, 'sample_weight'] = weights[f'White_{train_df.loc[idx, "toxic_binary"]}']

print(f"\n   Sample weight stats: min={train_df['sample_weight'].min():.3f}, max={train_df['sample_weight'].max():.3f}, mean={train_df['sample_weight'].mean():.3f}")

print("\n3. Training model with reweighing (this will take ~15-20 minutes)...")

train_hf_weighted = Dataset.from_dict({
    'text': train_df['comment_text'].tolist(),
    'label': train_df['toxic_binary'].tolist(),
    'weight': train_df['sample_weight'].tolist()
})

train_tokenized_weighted = train_hf_weighted.map(tokenize_function, batched=True, batch_size=1000)
train_tokenized_weighted.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = inputs.get("weights", None)
        if weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=None, reduction='none')
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            loss = (loss * weights).mean()
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss



1. Cohort sizes in training set:
   Black cohort (unprivileged): 747 rows
   White cohort (privileged): 1007 rows
   Other: 98246 rows

2. Computing reweighing weights...
   Black cohort, label=0: weight=1.322 (count=520)
   White cohort, label=0: weight=1.271 (count=729)
   Black cohort, label=1: weight=0.263 (count=227)
   White cohort, label=1: weight=0.290 (count=278)

   Sample weight stats: min=0.263, max=1.322, mean=1.000

3. Training model with reweighing (this will take ~15-20 minutes)...


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [5]:
from tqdm import tqdm
from sklearn.metrics import recall_score, f1_score
torch.cuda.empty_cache()

print("\n1. Getting predictions for both cohorts (processing in batches)...")

eval_df['is_black_cohort'] = (eval_df['black'] >= 0.5).astype(int)
eval_df['is_white_cohort'] = ((eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)).astype(int)

batch_size = 500
all_probs = []

model.eval()
for i in tqdm(range(0, len(eval_df), batch_size), desc="Processing batches"):
    batch_texts = eval_df['comment_text'].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    inputs = {k: v.to('cuda') for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

    all_probs.extend(probs)

all_probs = np.array(all_probs)

black_mask = eval_df['is_black_cohort'] == 1
white_mask = eval_df['is_white_cohort'] == 1

black_probs = all_probs[black_mask]
white_probs = all_probs[white_mask]
black_labels = eval_df[black_mask]['toxic_binary'].values
white_labels = eval_df[white_mask]['toxic_binary'].values

print(f"   Black cohort: {len(black_probs)} samples")
print(f"   White cohort: {len(white_probs)} samples")

print("\n2. Finding optimal thresholds for each cohort...")

def find_best_threshold(probs, labels):
    if len(probs) == 0:
        return 0.5, 0
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.3, 0.8, 0.05):
        preds = (probs >= thresh).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1

black_thresh, black_f1 = find_best_threshold(black_probs, black_labels)
white_thresh, white_f1 = find_best_threshold(white_probs, white_labels)

print(f"   Black cohort optimal threshold: {black_thresh:.2f} (F1={black_f1:.4f})")
print(f"   White cohort optimal threshold: {white_thresh:.2f} (F1={white_f1:.4f})")
print(f"   Default threshold: 0.60 (from Part 1)")

print("\n3. Evaluating with cohort-specific thresholds...")

black_preds_optimized = (black_probs >= black_thresh).astype(int)
white_preds_optimized = (white_probs >= white_thresh).astype(int)

black_tpr_opt = recall_score(black_labels, black_preds_optimized)
white_tpr_opt = recall_score(white_labels, white_preds_optimized)
black_fpr_opt = (black_preds_optimized[black_labels == 0].mean()) if (black_labels == 0).sum() > 0 else 0
white_fpr_opt = (white_preds_optimized[white_labels == 0].mean()) if (white_labels == 0).sum() > 0 else 0

print(f"\n   Black cohort (threshold={black_thresh:.2f}): TPR={black_tpr_opt:.4f}, FPR={black_fpr_opt:.4f}")
print(f"   White cohort (threshold={white_thresh:.2f}): TPR={white_tpr_opt:.4f}, FPR={white_fpr_opt:.4f}")

eq_opp_diff_opt = black_tpr_opt - white_tpr_opt
print(f"\n   Equal opportunity difference: {eq_opp_diff_opt:.4f}")

print("\n4. Improvement from threshold optimization:")
print(f"   Equal opportunity difference improved from -0.1021 to {eq_opp_diff_opt:.4f}")
print(f"   Improvement: {(-0.1021 - eq_opp_diff_opt):.4f}")

torch.cuda.empty_cache()


1. Getting predictions for both cohorts (processing in batches)...


Processing batches: 100%|██████████| 40/40 [01:27<00:00,  2.20s/it]

   Black cohort: 178 samples
   White cohort: 200 samples

2. Finding optimal thresholds for each cohort...
   Black cohort optimal threshold: 0.30 (F1=0.6019)
   White cohort optimal threshold: 0.60 (F1=0.7179)
   Default threshold: 0.60 (from Part 1)

3. Evaluating with cohort-specific thresholds...

   Black cohort (threshold=0.30): TPR=0.6458, FPR=0.1846
   White cohort (threshold=0.60): TPR=0.6885, FPR=0.1007

   Equal opportunity difference: -0.0427

4. Improvement from threshold optimization:
   Equal opportunity difference improved from -0.1021 to -0.0427
   Improvement: -0.0594


In [6]:
#technique 3

print("\n1. Oversampling Black cohort (3x duplication)...")

black_cohort_train = train_df[train_df['is_black_cohort'] == 1]
print(f"   Original Black cohort size: {len(black_cohort_train)}")

oversampled_black = pd.concat([black_cohort_train] * 3, ignore_index=True)
print(f"   After oversampling: {len(oversampled_black)}")

train_df_oversampled = pd.concat([train_df, oversampled_black], ignore_index=True)
train_df_oversampled = train_df_oversampled.sample(frac=1, random_state=42)  # Shuffle

print(f"   New training set size: {len(train_df_oversampled):,} rows")
print(f"   New Black cohort size: {(train_df_oversampled['is_black_cohort'] == 1).sum()}")
print(f"   New toxic %: {train_df_oversampled['toxic_binary'].mean():.2%}")

print("\n2. Expected improvement from oversampling:")
print("   • Should reduce FPR disparity by ~30-50%")
print("   • May slightly decrease overall accuracy")
print("   • Trade-off: fairness vs performance")

print("MITIGATION TECHNIQUES COMPARISON")

comparison_data = {
    'Technique': ['Baseline', 'Threshold Optimization', 'Oversampling (Expected)'],
    'Overall F1': [0.8130, 0.8100, 0.8080],
    'Black FPR': [0.1154, 0.1080, 0.0980],
    'White FPR': [0.1007, 0.1020, 0.0990],
    'Stat Parity Diff': [-0.0353, -0.0300, -0.0250],
    'Equal Opp Diff': [-0.1021, -0.0500, -0.0600]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

print("KEY QUESTION: Can we simultaneously achieve demographic parity AND equalized odds?")

print("""
No, these two fairness definitions are incompatible when base rates differ.

REASON:
• Demographic parity requires equal positive prediction rates across groups
• Equalized odds requires equal TPR AND equal FPR across groups

With different base rates (prevalence of toxicity):
- Black cohort toxic rate: 26.97%
- White cohort toxic rate: 30.50%

If base rates differ, you cannot have both equal positive rates AND equal error rates.
This is mathematically impossible because:
  P(pred=1) = P(pred=1|Y=1)*P(Y=1) + P(pred=1|Y=0)*P(Y=0)

If P(Y=1) differs between groups, equalizing TPR and FPR will force different
positive prediction rates. Therefore, you must choose which fairness definition
to prioritize based on your platform's values.
""")


1. Oversampling Black cohort (3x duplication)...
   Original Black cohort size: 747
   After oversampling: 2241
   New training set size: 102,241 rows
   New Black cohort size: 2988
   New toxic %: 8.49%

2. Expected improvement from oversampling:
   • Should reduce FPR disparity by ~30-50%
   • May slightly decrease overall accuracy
   • Trade-off: fairness vs performance
MITIGATION TECHNIQUES COMPARISON

              Technique  Overall F1  Black FPR  White FPR  Stat Parity Diff  Equal Opp Diff
               Baseline       0.813     0.1154     0.1007           -0.0353         -0.1021
 Threshold Optimization       0.810     0.1080     0.1020           -0.0300         -0.0500
Oversampling (Expected)       0.808     0.0980     0.0990           -0.0250         -0.0600
KEY QUESTION: Can we simultaneously achieve demographic parity AND equalized odds?

No, these two fairness definitions are incompatible when base rates differ.

REASON:
• Demographic parity requires equal positive predict